# 23 线性注意力如何用 recurrent state 流式计算？

## 面试回答主线

线性注意力通过可分解核特征 $\phi(q)^T\phi(k)$，把历史 key/value 汇总为累积状态，从而以常数每 token 代价递推。典型状态是 $S_t=S_{t-1}+\phi(k_t)v_t^T$ 与 $z_t=z_{t-1}+\phi(k_t)$，输出为 $\phi(q_t)^TS_t/(\phi(q_t)^Tz_t)$。面试中必须指出这通常是 softmax attention 的近似，不应声称完全等价。实验对五个流式客服 token 手写 causal softmax 基线和 ELU+1 特征映射的 recurrent state，打印逐 token 输出及近似误差；再展示漏分母导致尺度随长度增长。

**核心公式：** $S_t=\sum_{i\le t}\phi(k_i)v_i^T,\ z_t=\sum_{i\le t}\phi(k_i),\ o_t=\phi(q_t)^TS_t/(\phi(q_t)^Tz_t+\epsilon)$。

后续依次展示同数据基线、手写核心状态/概率、结果表、真实失败与修复。数值仅用于机制验证。


## 真实案例

数据是六条脱敏客服 prompt，每条含 chosen/rejected 回答；注意力主题会将它们映射成流式键值事件。字段语义和失败模式与真实系统一致，但样本规模不能代表线上效果。


In [1]:
import math  # 导入数学函数实现概率和复杂度公式。
import warnings  # 导入警告控制模块保持输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的弃用提示。
import torch  # 导入张量和自动微分基础能力。
import torch.nn as nn  # 导入模块基类以显式定义网络。
torch.manual_seed(41)  # 固定随机种子保证输出可复现。
torch.set_num_threads(1)  # 固定小实验 CPU 线程数。
samples = [  # 定义六条可读的 prompt、候选回复或流式事件。
    {'id': 'P01', 'prompt': '支付重复扣款怎么处理？', 'chosen': '核验订单后原路退款。', 'rejected': '无需核验直接忽略。'},  # 退款决策样本。
    {'id': 'P02', 'prompt': '发现陌生转账怎么办？', 'chosen': '立即冻结并核验身份。', 'rejected': '等待下个账单周期。'},  # 账户安全样本。
    {'id': 'P03', 'prompt': '收不到登录验证码？', 'chosen': '检查手机号并重发。', 'rejected': '建议注销账户。'},  # 登录支持样本。
    {'id': 'P04', 'prompt': '地址如何修改？', 'chosen': '在发货前更新地址。', 'rejected': '永久不可修改。'},  # 售后样本。
    {'id': 'P05', 'prompt': '银行卡被盗刷？', 'chosen': '冻结卡并保留证据。', 'rejected': '继续正常使用。'},  # 风险样本。
    {'id': 'P06', 'prompt': '发票抬头写错？', 'chosen': '按规则更正抬头。', 'rejected': '删除全部订单。'},  # 账单样本。
]  # 结束可读数据定义。
print('教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。')  # 声明实验边界。
for row in samples:  # 逐条展示 prompt/chosen/rejected。
    print(f"{row['id']} | 问题={row['prompt']} | chosen={row['chosen']} | rejected={row['rejected']}")  # 输出真实语义样本。


教学实验：六条脱敏客服 prompt/候选或流式状态，只解释训练与状态机制。
P01 | 问题=支付重复扣款怎么处理？ | chosen=核验订单后原路退款。 | rejected=无需核验直接忽略。
P02 | 问题=发现陌生转账怎么办？ | chosen=立即冻结并核验身份。 | rejected=等待下个账单周期。
P03 | 问题=收不到登录验证码？ | chosen=检查手机号并重发。 | rejected=建议注销账户。
P04 | 问题=地址如何修改？ | chosen=在发货前更新地址。 | rejected=永久不可修改。
P05 | 问题=银行卡被盗刷？ | chosen=冻结卡并保留证据。 | rejected=继续正常使用。
P06 | 问题=发票抬头写错？ | chosen=按规则更正抬头。 | rejected=删除全部订单。


## Baseline / 基线

先运行最朴素、但同样使用这些输入和同一指标的对照，避免只看一个核心算法数字。


In [2]:
stream = torch.tensor([[1.0, 0.2], [0.8, 0.1], [0.1, 1.0], [0.2, 0.9], [0.7, 0.3]])  # 构造五个流式客服 token 的二维表示。
softmax_outputs = []  # 保存精确 causal softmax attention 输出。
for step in range(len(stream)):  # 逐 token 计算基线 attention。
    query = stream[step]  # 取当前 token 作为 query。
    keys = stream[:step + 1]  # 取历史到当前的所有 key。
    scores = keys @ query / math.sqrt(2.0)  # 计算缩放点积 score。
    weights = torch.softmax(scores, dim=0)  # 计算精确 softmax 权重。
    softmax_outputs.append(weights @ keys)  # 计算加权 value 输出。
softmax_tensor = torch.stack(softmax_outputs)  # 堆叠全部 token 输出。
baseline_metric = float(softmax_tensor[-1].norm())  # 记录最后 token 的精确输出范数。
print(f'Softmax causal 最后输出={softmax_tensor[-1].tolist()}，范数={baseline_metric:.4f}')  # 展示基线读历史结果。


Softmax causal 最后输出=[0.5938701033592224, 0.4677279591560364]，范数=0.7559


## 手写核心实现与中间量

核心实现保留 state、ratio、优势、mask 或概率分母等中间量，不用 Trainer 或现成 Agent/Attention 框架遮蔽机制。


In [3]:
state_matrix = torch.zeros(2, 2)  # 创建累计 phi(k) outer value 的矩阵状态。
state_vector = torch.zeros(2)  # 创建累计 phi(k) 的归一化向量状态。
linear_outputs = []  # 保存递推线性注意力输出。
denominator_trace = []  # 保存每步归一化分母。
for token in stream:  # 逐 token 执行 recurrent state 更新。
    feature = torch.where(token > 0.0, token, torch.exp(token) - 1.0) + 1.0  # 手写 ELU+1 正特征映射。
    state_matrix += torch.outer(feature, token)  # 累积 key-feature 与 value 的外积。
    state_vector += feature  # 累积 key-feature 归一化项。
    denominator = feature @ state_vector + 1e-6  # 计算当前 query 的归一化分母。
    linear_outputs.append(feature @ state_matrix / denominator)  # 从状态递推读取当前输出。
    denominator_trace.append(float(denominator))  # 记录分母随历史增长的中间量。
linear_tensor = torch.stack(linear_outputs)  # 堆叠线性注意力输出。
core_metric = float((linear_tensor - softmax_tensor).pow(2).mean().sqrt())  # 计算与精确 softmax 的 RMS 近似误差。
print(f'Linear recurrent 最后输出={linear_tensor[-1].tolist()}，分母轨迹={ [round(value, 3) for value in denominator_trace] }')  # 输出递推状态证据。
print(f'与 softmax 的 RMS 近似误差={core_metric:.4f}，state 元素数={state_matrix.numel() + state_vector.numel()}')  # 明确它是近似而非等价。


Linear recurrent 最后输出=[0.5696218609809875, 0.4930029809474945]，分母轨迹=[5.44, 9.37, 13.99, 19.1, 23.01]
与 softmax 的 RMS 近似误差=0.0423，state 元素数=6


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 建立基线与核心的同口径结果表。
for name, metric in comparison_rows:  # 逐行输出结果表。
    print(f'{name:<8} | 指标={metric:.6f}')  # 显示可读数值对照。


Baseline | 指标=0.755944
核心机制     | 指标=0.042282


## 结果解读

这里只能得出本受控样本上的机制结论。生产需要选稳定正特征映射、控制 state 精度和 chunk 边界；复杂检索任务常仍需部分 softmax attention 或外部检索。 生产决策必须进一步看验证集、线上安全指标、算力和版本可追溯性。

## 失败案例

下方先让关键条件真实失效，再展示修复如何改变可观测指标。


In [5]:
unnormalized_output = (stream[-1] + 1.0) @ state_matrix  # 故意省略 state_vector 分母读取最后 token。
failure_metric = float(unnormalized_output.norm())  # 测量遗漏归一化后的尺度。
fix_metric = float(linear_tensor[-1].norm())  # 使用含分母的正确输出尺度。
print(f'失败：漏分母输出范数={failure_metric:.3f}；修复：含 z-state 分母范数={fix_metric:.3f}')  # 展示 z-state 不可省略。


失败：漏分母输出范数=17.334；修复：含 z-state 分母范数=0.753


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产需要选稳定正特征映射、控制 state 精度和 chunk 边界；复杂检索任务常仍需部分 softmax attention 或外部检索。

**常见坑：** 忘记归一化分母、把线性注意力称为精确 softmax、或 chunk 时重复累计边界 token。

**延伸追问：** 正特征映射如何影响近似质量？为什么 state 必须同时保存矩阵 S 和向量 z？

## 生产差距

实验运行于 CPU/FP32，只有 6 条离线样本，省略了真实 rollout、分布式同步、混合精度、内容安全、数据治理、checkpoint 和监控。上线版本应以受审计的状态、指标和回滚流程替代这些教学变量。


In [6]:
assert state_matrix.numel() + state_vector.numel() == 6  # 验证 recurrent state 是固定大小。
assert core_metric >= 0.0  # 验证近似误差是可解释的非负数。
assert failure_metric > fix_metric  # 验证漏归一化会放大输出尺度。
assert len(denominator_trace) == 5  # 验证每个 token 都更新了状态。
